# Zerobus Ingest Benchmark Driver

Run all four ingest patterns (gRPC sync, gRPC async, HTTP sync, HTTP async) in a loop
using `zbhelper.ingest_benchmark`. Each iteration ingests `_N` rows and prints the
full Step-4d metrics block.

Set `SP_NAME` and `TABLE` in **Step 2** — workspace URL, region, ZeroBus endpoint,
service principal, and OAuth secret are discovered and provisioned automatically.
Run all cells top to bottom.

### Step 1: Install dependencies

In [1]:
%pip install --quiet aiohttp requests


Note: you may need to restart the kernel to use updated packages.


In [2]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

_nb = next((str(p) for p in [Path.cwd(), Path.cwd() / "notebooks"] if (p / "zbhelper").is_dir()), None)
if _nb and _nb not in sys.path:
    sys.path.insert(0, _nb)

import zbhelper.ingest_benchmark as zb
import zbhelper.setup as zbsetup
print("zbhelper loaded")


zbhelper loaded


### Step 2: Connection & table configuration

`zbhelper.setup` auto-discovers the workspace URL, region, and ZeroBus endpoint, then
ensures the service principal and OAuth secret exist. Set `SP_NAME` and optionally
`TABLE` below — everything else is derived automatically.


In [3]:
# ── User-configurable ────────────────────────────────────────────────────
SP_NAME = "lfcdemo_zerobus"       # service principal display name (also drives secret scope)
TABLE   = "airquality_benchmark"  # table segment; catalog/schema auto-detected from workspace

# ── Step 2.a: auto-discover workspace + ZeroBus endpoint ─────────────────
_ws = zbsetup.discover_workspace(dbutils)
DATABRICKS_WORKSPACE_URL = _ws["workspace_url"]
DATABRICKS_WORKSPACE_ID  = _ws["workspace_id"]
ZEROBUS_INGEST_URL       = _ws["zerobus_ingest_url"]
SERVER_ENDPOINT          = _ws["server_endpoint"]

# ── Step 2.b + 2.c: ensure SP exists; mint/validate OAuth secret ──────────
_sp = zbsetup.ensure_service_principal(_ws, dbutils, sp_name=SP_NAME)
CLIENT_ID     = _sp["client_id"]
CLIENT_SECRET = _sp["client_secret"]

# ── Step 2.d + 3: create UC table, grant SP permissions ───────────────────
_tbl = zbsetup.ensure_table(spark, _ws, CLIENT_ID, table=TABLE)
CATALOG    = _tbl["catalog"]
SCHEMA     = _tbl["schema"]
TABLE_NAME = _tbl["table_name"]


Cleared metadata-service auth env vars (Connect kernel).
workspace_url=https://e2-demo-field-eng.cloud.databricks.com
workspace_id=1444828305810485
region=us-west-2
server_endpoint=https://1444828305810485.zerobus.us-west-2.cloud.databricks.com
SP exists: 'lfcdemo_zerobus_sp' sp_id=75332893425169
OAuth client secret is valid.
CATALOG auto-detected: 'main'
SCHEMA auto-detected: 'robert_lee'
Table ready: main.robert_lee.airquality_benchmark


### Step 3: Benchmark settings

In [4]:
_N       = 1000          # total rows per iteration
_SINGLES = min(10, _N)  # single-row phase (4a) row count
_ITERS   = 1             # number of full benchmark loops

# Set False to skip a pattern
_RUN_GRPC_SYNC  = True
_RUN_GRPC_ASYNC = True
_RUN_HTTP_SYNC  = True
_RUN_HTTP_ASYNC = True


### Step 4: Benchmark loop

In [5]:
_benchmark_rows = []  # accumulates one dict per (protocol, iteration) for CSV export

for _iteration in range(1, _ITERS + 1):
    if _ITERS > 1:
        print(f"\n{'='*60}")
        print(f"Iteration {_iteration} / {_ITERS}")
        print(f"{'='*60}")

    records_4a = zb.build_records(_SINGLES)
    records_4b = zb.build_records(_N - _SINGLES, offset=_SINGLES)

    # ------------------------------------------------------------------ #
    # gRPC sync                                                            #
    # ------------------------------------------------------------------ #
    if _RUN_GRPC_SYNC:
        print("\n--- gRPC sync ---")
        baseline = zb.fetch_row_baseline(spark, TABLE_NAME)
        stream, stream_open_s = zb.open_grpc_stream_sync(
            SERVER_ENDPOINT, DATABRICKS_WORKSPACE_URL,
            CLIENT_ID, CLIENT_SECRET, TABLE_NAME,
        )
        singles = zb.ingest_singles_grpc_sync(stream, records_4a)
        batch   = zb.ingest_batch_and_close_grpc_sync(stream, records_4b, stream_open_s=stream_open_s)
        vis     = zb.poll_visibility(
            spark, TABLE_NAME, baseline["count"] + _N,
            batch.t_after_close, singles.t_4a0,
        )
        zb.print_metrics(TABLE_NAME, _N, baseline["count"], baseline, singles, batch, vis)
        _benchmark_rows.append(zb.to_csv_row("grpc_sync", _iteration, _N, singles, batch, vis))

    # ------------------------------------------------------------------ #
    # gRPC async                                                           #
    # ------------------------------------------------------------------ #
    if _RUN_GRPC_ASYNC:
        print("\n--- gRPC async ---")
        baseline = zb.fetch_row_baseline(spark, TABLE_NAME)
        stream, stream_open_s = await zb.open_grpc_stream_async(
            SERVER_ENDPOINT, DATABRICKS_WORKSPACE_URL,
            CLIENT_ID, CLIENT_SECRET, TABLE_NAME,
        )
        singles = await zb.ingest_singles_grpc_async(stream, records_4a)
        batch   = await zb.ingest_batch_and_close_grpc_async(stream, records_4b, stream_open_s=stream_open_s)
        vis     = zb.poll_visibility(
            spark, TABLE_NAME, baseline["count"] + _N,
            batch.t_after_close, singles.t_4a0,
        )
        zb.print_metrics(TABLE_NAME, _N, baseline["count"], baseline, singles, batch, vis)
        _benchmark_rows.append(zb.to_csv_row("grpc_async", _iteration, _N, singles, batch, vis))

    # ------------------------------------------------------------------ #
    # HTTP sync                                                            #
    # ------------------------------------------------------------------ #
    if _RUN_HTTP_SYNC:
        import requests as _requests
        print("\n--- HTTP sync ---")
        baseline = zb.fetch_row_baseline(spark, TABLE_NAME)
        token      = zb.fetch_http_token(
            DATABRICKS_WORKSPACE_URL, DATABRICKS_WORKSPACE_ID,
            CLIENT_ID, CLIENT_SECRET, CATALOG, SCHEMA, TABLE_NAME,
        )
        insert_url = zb.http_insert_url(ZEROBUS_INGEST_URL, TABLE_NAME)
        with _requests.Session() as _http_session:
            singles = zb.ingest_singles_http_sync(insert_url, token, records_4a, session=_http_session)
            batch   = zb.ingest_batch_http_sync(insert_url, token, records_4b, session=_http_session)
        vis = zb.poll_visibility(
            spark, TABLE_NAME, baseline["count"] + _N,
            batch.t_after_close, singles.t_4a0,
        )
        zb.print_metrics(TABLE_NAME, _N, baseline["count"], baseline, singles, batch, vis)
        _benchmark_rows.append(zb.to_csv_row("http_sync", _iteration, _N, singles, batch, vis))

    # ------------------------------------------------------------------ #
    # HTTP async                                                           #
    # ------------------------------------------------------------------ #
    if _RUN_HTTP_ASYNC:
        import aiohttp as _aiohttp
        print("\n--- HTTP async ---")
        baseline = zb.fetch_row_baseline(spark, TABLE_NAME)
        token      = zb.fetch_http_token(
            DATABRICKS_WORKSPACE_URL, DATABRICKS_WORKSPACE_ID,
            CLIENT_ID, CLIENT_SECRET, CATALOG, SCHEMA, TABLE_NAME,
        )
        insert_url = zb.http_insert_url(ZEROBUS_INGEST_URL, TABLE_NAME)
        async with _aiohttp.ClientSession() as _http_session:
            singles = await zb.ingest_singles_http_async(insert_url, token, records_4a, session=_http_session)
            batch   = await zb.ingest_batch_http_async(insert_url, token, records_4b, session=_http_session)
        vis = zb.poll_visibility(
            spark, TABLE_NAME, baseline["count"] + _N,
            batch.t_after_close, singles.t_4a0,
        )
        zb.print_metrics(TABLE_NAME, _N, baseline["count"], baseline, singles, batch, vis)
        _benchmark_rows.append(zb.to_csv_row("http_async", _iteration, _N, singles, batch, vis))



--- gRPC sync ---
2026-04-16T22:04:47.959328Z  INFO databricks_zerobus_ingest_sdk: Successfully created stream stream_id=ab835707-83a0-4a44-995d-7b791e0c562c
2026-04-16T22:04:47.959495Z  INFO databricks_zerobus_ingest_sdk: Successfully created stream stream_id=ab835707-83a0-4a44-995d-7b791e0c562c
2026-04-16T22:04:47.959548Z  INFO databricks_zerobus_ingest_sdk: Successfully created new ephemeral stream stream_id=ab835707-83a0-4a44-995d-7b791e0c562c
2026-04-16T22:04:47.960817Z  INFO databricks_zerobus_ingest_sdk: Stream is not caught up to any offset yet. Waiting for the first offset. stream_id=ab835707-83a0-4a44-995d-7b791e0c562c
[ack callback] offset 0 acknowledged
2026-04-16T22:04:48.155746Z  INFO databricks_zerobus_ingest_sdk: Stream is caught up to the given offset. Waiting for acknowledgement completed. stream_id=ab835707-83a0-4a44-995d-7b791e0c562c
2026-04-16T22:04:48.157075Z  INFO databricks_zerobus_ingest_sdk: Stream is caught up to offset 0. Waiting for offset 1. stream_id=ab8

### Step 5: Export results to CSV

Writes `benchmark_results.csv` next to the notebook (or repo root when running locally in Cursor).
Open in Excel / Google Sheets — suggested charts:
- **Bar chart**: `single_total_mean_ms` or `batch_ms_per_row` grouped by `protocol`
- **Line chart** (multi-iteration runs): metric vs `iteration`, series per `protocol`
- **Stacked bar** (gRPC only): `single_send_mean_ms` + `single_ack_mean_ms` to show send vs wait split

In [ ]:
import os
_csv_path = os.path.join(os.path.dirname(os.path.abspath(__vsc_ipynb_file__)) if "__vsc_ipynb_file__" in vars() else os.getcwd(), "benchmark_results.csv")
_csv_text = zb.write_csv(_benchmark_rows, path=_csv_path)
print(f"Wrote {len(_benchmark_rows)} rows → {_csv_path}\n")
print(_csv_text)